# Autonomous Systems Portfolio 1  

|Name     |Studentnumber|Github    |
|---------|-------------|----------|
|Henry Lau|22122958     |HenryLau08|
|Michal|||
|Mohamed|||

Game: Connect 4

In [1]:
pip install pettingzoo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.5/852.5 kB 13.5 MB/s eta 0:00:00


In [19]:
import numpy as np
from pettingzoo.classic import connect_four_v3


# -----------------------------
# Initialize environment
# -----------------------------
env = connect_four_v3.env()
env.reset(seed=42)


# -----------------------------
# Convert Observation
# -----------------------------
def convert_observation(obs):
    board = np.zeros((6, 7), dtype=int)

    for i in range(6):
        for j in range(7):
            if obs[i, j, 0] == 1:
                board[i, j] = 1
            elif obs[i, j, 1] == 1:
                board[i, j] = 2

    return board


# -----------------------------
# Print Board (Improved UX)
# -----------------------------
def print_board(board, mask):
    print("\n" + "=" * 20)
    print("Current board:\n")

    for i in range(6):
        row = []
        for j in range(7):
            if board[i, j] == 1:
                row.append("X")
            elif board[i, j] == 2:
                row.append("O")
            else:
                row.append(".")
        print(" ".join(row))

    print("\nColumns:")
    print(" ".join([str(i) if mask[i] else "x" for i in range(7)]))
    print("=" * 20)


# -----------------------------
# Helpers
# -----------------------------
def drop_piece(board, col, player):
    new_board = board.copy()

    for row in range(5, -1, -1):
        if new_board[row, col] == 0:
            new_board[row, col] = player
            return new_board

    return None


def check_win(board, player):
    rows, cols = board.shape

    # Horizontal
    for r in range(rows):
        for c in range(cols - 3):
            if all(board[r, c + i] == player for i in range(4)):
                return True

    # Vertical
    for r in range(rows - 3):
        for c in range(cols):
            if all(board[r + i, c] == player for i in range(4)):
                return True

    # Diagonal \
    for r in range(rows - 3):
        for c in range(cols - 3):
            if all(board[r + i, c + i] == player for i in range(4)):
                return True

    # Diagonal /
    for r in range(3, rows):
        for c in range(cols - 3):
            if all(board[r - i, c + i] == player for i in range(4)):
                return True

    return False


def get_opponent(player):
    return 2 if player == 1 else 1


# -----------------------------
# Winning / Blocking
# -----------------------------
def find_winning_move(board, mask, player):
    for col in np.where(mask)[0]:
        temp = drop_piece(board, col, player)
        if temp is not None and check_win(temp, player):
            return col
    return None


def find_blocking_move(board, mask, player):
    return find_winning_move(board, mask, get_opponent(player))


# -----------------------------
# Fork Logic
# -----------------------------
def count_winning_moves(board, mask, player):
    count = 0
    for col in np.where(mask)[0]:
        temp = drop_piece(board, col, player)
        if temp is not None and check_win(temp, player):
            count += 1
    return count


def find_fork_move(board, mask, player):
    for col in np.where(mask)[0]:
        temp = drop_piece(board, col, player)
        if temp is None:
            continue

        if count_winning_moves(temp, mask, player) >= 2:
            return col
    return None


def find_blocking_fork(board, mask, player):
    return find_fork_move(board, mask, get_opponent(player))


# -----------------------------
# Human Player
# -----------------------------
def get_human_action(observation):
    mask = observation["action_mask"]
    board = observation["board"]

    print_board(board, mask)

    while True:
        try:
            move = int(input("Choose column (0-6): "))
            if 0 <= move < 7 and mask[move]:
                return move
            print("Invalid move, try again.")
        except:
            print("Enter a valid number.")


# -----------------------------
# AI Strategy
# -----------------------------
def smart_strategy(observation, agent):
    board = observation["board"]
    mask = observation["action_mask"]

    player = 1 if agent == "player_0" else 2

    # 1. Win
    move = find_winning_move(board, mask, player)
    if move is not None:
        return int(move)

    # 2. Block
    move = find_blocking_move(board, mask, player)
    if move is not None:
        return int(move)

    # 3. Fork
    move = find_fork_move(board, mask, player)
    if move is not None:
        return int(move)

    # 4. Block fork
    move = find_blocking_fork(board, mask, player)
    if move is not None:
        return int(move)

    # 5. Center
    valid = np.where(mask)[0]
    if 3 in valid:
        return 3

    # 6. Random
    return int(np.random.choice(valid))


# -----------------------------
# Game Loop
# -----------------------------
game_over = False

for agent in env.agent_iter():
    obs, reward, term, trunc, _ = env.last()

    # Dead agent -> must pass None
    if term or trunc:
        if not game_over:
            game_over = True
            if reward == 1:
                print("\nYou win!" if agent == "player_0" else "\nComputer wins!")
            elif reward == 0:
                print("\nDraw!")
        action = None  # MUST be None for dead agents

    else:
        # Alive agent -> compute action
        board = convert_observation(obs["observation"])
        observation = {
            "board": board,
            "action_mask": obs["action_mask"]
        }

        if agent == "player_0":
            action = get_human_action(observation)
        else:
            action = smart_strategy(observation, agent)

    env.step(action)

env.close()


Current board:

. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .

Columns:
0 1 2 3 4 5 6
Choose column (0-6): 2

Current board:

. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . X O . . .

Columns:
0 1 2 3 4 5 6
Choose column (0-6): 3

Current board:

. . . . . . .
. . . . . . .
. . . . . . .
. . . O . . .
. . . X . . .
. . X O . . .

Columns:
0 1 2 3 4 5 6
Choose column (0-6): 2

Current board:

. . . . . . .
. . . . . . .
. . . O . . .
. . . O . . .
. . X X . . .
. . X O . . .

Columns:
0 1 2 3 4 5 6
Choose column (0-6): 2

Current board:

. . . . . . .
. . . . . . .
. . O O . . .
. . X O . . .
. . X X . . .
. . X O . . .

Columns:
0 1 2 3 4 5 6
Choose column (0-6): 4

Current board:

. . . . . . .
. . . O . . .
. . O O . . .
. . X O . . .
. . X X . . .
. . X O X . .

Columns:
0 1 2 3 4 5 6
Choose column (0-6): 3

Current board:

. . . X . . .
. . . O . . .
. . O O . . .
. . X O . . .
. . X X . . .
. . X O X O .

Columns:
0